# HEILO Seed Code v0.2: continua o treino do modelo de código

**Resultado real da v0.1:** 80 min de treino e perplexidade 6,9 em código novo. Ela escreve Python válido (81%), mas ainda **não acerta a lógica**: pass@1 = 0 de 20. Por exemplo, para `soma(a, b)` ela escreveu `return soma(a, b)`.

A avaliação da v0.1 tinha um bug: ela cortava todo o corpo das funções. Isso já foi corrigido, e os números acima são os corrigidos.

**O que muda na v0.2**
- Continua da v0.1, sem recomeçar, com um corpus **2,4× maior** de código Python público.
- Mais 150 minutos de GPU.

Pode rodar ao mesmo tempo que o notebook do Seed v1.1. **Antes:** GPU T4 e o secret `GITHUB_TOKEN`. Depois, `Executar tudo`.

In [ ]:
# 1) Configuração
REPO = "hugolindo12/HELIO"
BRANCH = "main"
NOME_VERSAO = "v0.2"
MINUTOS_PRETREINO = 150
MAX_CARACTERES = 1_200_000_000
LOTE = 32
import torch, time, os, sys, json, shutil, subprocess
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NENHUMA — ative a GPU!")

In [ ]:
# 2) Drive (mesma pasta da v0.1)
from google.colab import drive
drive.mount("/content/drive")
PASTA = "/content/drive/MyDrive/heilo_seed_code"
assert os.path.exists(f"{PASTA}/tokenizer.json") and os.path.exists(f"{PASTA}/pretreino.pt"), \
    "Não achei o tokenizer/checkpoint da v0.1 no Drive (pasta heilo_seed_code)."

In [ ]:
# 3) Código da HEILO (pasta própria para não esbarrar no outro notebook)
from google.colab import userdata
TOKEN = userdata.get("GITHUB_TOKEN")
URL = f"https://{TOKEN}@github.com/{REPO}.git"
BASE = "/content/code_run"
if not os.path.exists(f"{BASE}/heilo"):
    os.makedirs(BASE, exist_ok=True)
    subprocess.run(["git", "clone", "-b", BRANCH, URL, f"{BASE}/heilo"], check=True)
else:
    subprocess.run(f"cd {BASE}/heilo && git stash -u -q; git pull -q --rebase", shell=True)
sys.path.insert(0, BASE)
!pip -q install datasets tokenizers

In [ ]:
# 4) Corpus de código maior (mesmo tokenizer da v0.1)
from pathlib import Path
from heilo.training import seed_code, pretrain
from heilo.models.seed.tokenizer import BPETokenizer
tok = BPETokenizer.from_json(Path(PASTA, "tokenizer.json").read_text(encoding="utf-8"))
corpus_dir = Path(PASTA) / "corpus_grande"
if not (corpus_dir / "meta.json").exists():
    t0 = time.time()
    meta = pretrain.preparar_corpus(seed_code.textos_python(MAX_CARACTERES), tok, corpus_dir,
                                    max_tokens=10**12, log=lambda s: None)
    print(meta, f"{(time.time()-t0)/60:.1f} min")
else:
    print(open(corpus_dir / "meta.json").read())

In [ ]:
# 5) Continua o pré-treino da v0.1 (guarda uma cópia dela antes)
ck = Path(PASTA) / "pretreino.pt"
copia = Path(PASTA) / "pretreino_v0.1.pt"
if not copia.exists():
    shutil.copy(ck, copia)
arq_passos = Path(PASTA, "passos_v0.2.txt")
if not arq_passos.exists():
    st = torch.load(copia, map_location="cpu", weights_only=False)
    ultimo = st["hist"][-1]
    total = st["passo"] + int(MINUTOS_PRETREINO * 60 / (ultimo["min"] * 60 / ultimo["passo"]))
    arq_passos.write_text(str(total)); del st
PASSOS = int(arq_passos.read_text())
cfg = seed_code.CONFIG_CODE
res = pretrain.pretreinar(cfg, tok, corpus_dir, ck, passos=PASSOS, lote=LOTE, lr=3e-4, lr_min=3e-5,
                          tempo_max_min=MINUTOS_PRETREINO + 20, avaliar_cada=500, salvar_cada=1000)
modelo = res["modelo"]
print("pré-treino:", res["passo"], "passos |", res["hist"][-1])

In [ ]:
# 6) Avaliação REAL (avaliação corrigida): completa funções e EXECUTA os testes
av = seed_code.avaliar_codigo(modelo)
print({k: v for k, v in av.items() if k != "detalhes"})
for d in av["detalhes"][:5]:
    print("-" * 40); print(d["codigo"])

In [ ]:
# 7) Exportar + salvar no GitHub
from heilo.training.records import agora
destino = Path(BASE) / "heilo" / "models" / "seed_code" / "versions" / NOME_VERSAO
destino.mkdir(parents=True, exist_ok=True)
pretrain.exportar(modelo, destino / "heilo_seed_code.pt",
                  {"versao": NOME_VERSAO, "base": "v0.1", "passos": res["passo"], "hist": res["hist"],
                   "corpus": json.loads((corpus_dir / "meta.json").read_text())})
manifesto = {"versao": NOME_VERSAO, "base": "v0.1", "criada": agora(), "tipo": "HEILO Seed Code (do zero)",
             "config": cfg.__dict__, "passos": res["passo"], "minutos": res["minutos"],
             "ultima_validacao": res["hist"][-1] if res["hist"] else None,
             "avaliacao": {k: v for k, v in av.items() if k != "detalhes"},
             "comparacao_v0.1": {"pass@1": 0.0, "pass@6": 0.0, "codigo_valido": 0.81, "ppl_val": 6.93},
             "corpus": "codeparrot/codeparrot-clean (código Python público do GitHub; licenças variadas)"}
(destino / "manifest.json").write_text(json.dumps(manifesto, indent=2, ensure_ascii=False))
(destino / "avaliacao.json").write_text(json.dumps(av, indent=2, ensure_ascii=False))
print(json.dumps(manifesto["avaliacao"], ensure_ascii=False), "| ppl:", manifesto["ultima_validacao"])
!cd {BASE}/heilo && git config user.email "heilo@colab" && git config user.name "HEILO (Colab)"
!cd {BASE}/heilo && git add -f models/seed_code && git commit -q -m "HEILO Seed Code {NOME_VERSAO} (Colab)" || echo nada
!cd {BASE}/heilo && git stash -u -q; git pull --rebase && git push && echo "ENVIADO PARA O GITHUB"